# Raster Validation

Validates raster built-up datasets (TEMPO, GHSL Built-S/V/H, Google OBT) against reference building footprints.

Pipeline per city:
1. Tile the AOI into 1 km × 1 km cells
2. For each enabled raster candidate, read the year-specific file (`{city_slug}_{name}_{year}.tif`)
3. Rasterize reference footprints onto the candidate grid (fractional coverage via oversampling)
4. Compute tile-level binary (TP/FP/FN/F1) and area-based metrics
5. Save tile metrics, city summary, and figures to `outputs/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose raster outputs already exist.
# When False (default), cities with an existing sentinel parquet are skipped automatically.
OVERWRITE = True

In [4]:
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
# import rasterio
# from pathlib import Path

# tile_path = Path(f"{PROJECT_ROOT}/data/WSFsoftRelease/Oceania/WSFtracker_20160701-20250701_-126_-26.tif")

# with rasterio.open(tile_path) as src:
#     print("CRS:", src.crs)
#     print("Bounds:", src.bounds)
#     print("Shape:", (src.height, src.width))
#     print("Resolution:", src.res)

In [6]:
# from glob import glob

# paths = glob(f"{PROJECT_ROOT}/data/WSFsoftRelease/**/*.tif", recursive=True)[:5]

# for p in paths:
#     with rasterio.open(p) as src:
#         print(p.split("/")[-1], "->", src.bounds)

In [7]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

raster_datasets = _cfg_preview.get("raster", {}).get("datasets", [])
enabled = [
    f"{d['name']}" + (f"_{d['year']}" if d.get("year") is not None else "")
    for d in raster_datasets
    if d.get("enabled", True)
]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled raster datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled raster datasets: ['obt_2023', 'tempo_2023q4', 'ghsl_built_s_2025', 'wsf_tracker']


In [8]:
# Patch overwrite flag and instantiate the Validator.
# This reads the config and AOI tracker, resolves file paths,
# and logs how many city datasets are queued.
import tempfile

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nCities queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

20:40:35  INFO      Validation tracker: 52 -> 52 suitable rows.
20:40:36  INFO      Loaded 52 dataset(s) for validation.
20:40:36  INFO      Loaded 52 dataset(s) for validation.



Cities queued: 52
  usa-brentwood
  usa-sandiego
  usa-boise
  usa-lasvegas
  usa-saltlakecity
  usa-permianbasin
  mex-mexicocity
  usa-bentonville
  usa-gonzales
  usa-atlanta
  usa-portstlucie
  pan-panamacity
  usa-raleigh
  jam-kingston
  usa-allentown
  per-cusco
  chl-calama
  bra-manaus
  bra-saopaulo
  sen-dakar
  alg-tindouf
  gha-kumasi
  lby-benghazi
  zmb-lusaka
  zaf-durban
  egy-cairo
  sdn-khartoum
  uga-kampala
  gbr-birmingham
  gbr-london
  nld-rotterdam
  rou-bucharest
  yem-dhamar
  sau-riyadh
  kwt-kuwaitcity
  rus-astrakhan
  are-abudhabi
  uzb-zarafshan
  ind-mumbai
  ind-tada
  ind-vijayawada
  bgd-dhaka
  chn-chengdu
  chn-zhuhai
  chn-guangzhou
  chn-wuhan
  chn-lujiang
  chn-yangzhou
  phl-angelescity
  chn-shanghai
  kor-sejong
  aus-melbourne


In [9]:
# Preview: show which raster files will be looked up for each city × dataset combination.
# Files that don't exist on disk are flagged so you can fix paths before running.
import pandas as pd
from pathlib import Path

data_dir = Path(_cfg_patched.get("root_dir", str(PROJECT_ROOT))) / _cfg_patched.get("data_dir", "data/01_raw")

preview_rows = []
for ds in v.datasets:
    city_slug    = ds["id"].lower()
    city_slug_us = city_slug.replace("-", "_")
    rast_dir     = data_dir / ds["id"] / "raster"

    for cand in _cfg_patched.get("raster", {}).get("datasets", []):
        if not cand.get("enabled", True):
            continue
        ds_name = cand["name"].replace("-", "_")
        year    = cand.get("year")
        if year is not None:
            fpath = rast_dir / f"{city_slug_us}_{ds_name}_{year}.tif"
        else:
            matches = sorted(rast_dir.glob(f"{city_slug_us}_{ds_name}*.tif"))
            fpath   = matches[0] if matches else rast_dir / f"{city_slug_us}_{ds_name}_?.tif"
        preview_rows.append({
            "city":    ds["id"],
            "dataset": f"{ds_name}_{year}" if year is not None else ds_name,
            "file":    fpath.name,
            "exists":  fpath.exists(),
        })

preview_df = pd.DataFrame(preview_rows)
missing = preview_df[~preview_df["exists"]]
print(f"Total city × dataset pairs: {len(preview_df)}")
print(f"Missing files:              {len(missing)}")
display(preview_df)

Total city × dataset pairs: 208
Missing files:              27


,city,dataset,file,exists
0,usa-brentwood,obt_2023,usa_brentwood_obt_2023.tif,False
1,usa-brentwood,tempo_2023q4,usa_brentwood_tempo_2023q4.tif,True
2,usa-brentwood,ghsl_built_s_2025,usa_brentwood_ghsl_built_s_2025.tif,True
3,usa-brentwood,wsf_tracker,usa_brentwood_wsf_tracker.tif,True
4,usa-sandiego,obt_2023,usa_sandiego_obt_2023.tif,True
...,...,...,...,...
203,kor-sejong,wsf_tracker,kor_sejong_wsf_tracker.tif,True
204,aus-melbourne,obt_2023,aus_melbourne_obt_2023.tif,False
205,aus-melbourne,tempo_2023q4,aus_melbourne_tempo_2023q4.tif,True
206,aus-melbourne,ghsl_built_s_2025,aus_melbourne_ghsl_built_s_2025.tif,True


In [10]:
preview_df

,city,dataset,file,exists
0,usa-brentwood,obt_2023,usa_brentwood_obt_2023.tif,False
1,usa-brentwood,tempo_2023q4,usa_brentwood_tempo_2023q4.tif,True
2,usa-brentwood,ghsl_built_s_2025,usa_brentwood_ghsl_built_s_2025.tif,True
3,usa-brentwood,wsf_tracker,usa_brentwood_wsf_tracker.tif,True
4,usa-sandiego,obt_2023,usa_sandiego_obt_2023.tif,True
...,...,...,...,...
203,kor-sejong,wsf_tracker,kor_sejong_wsf_tracker.tif,True
204,aus-melbourne,obt_2023,aus_melbourne_obt_2023.tif,False
205,aus-melbourne,tempo_2023q4,aus_melbourne_tempo_2023q4.tif,True
206,aus-melbourne,ghsl_built_s_2025,aus_melbourne_ghsl_built_s_2025.tif,True


In [11]:
results = v.validate_raster()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"city": k, "status": "ok" if ok else "failed"} for k, ok in results.items()]
)
print(f"\nDone — {len(summary)} cities processed.\n")
display(summary.groupby("status")["city"].count().rename("count").to_frame())
display(summary)

20:40:36  INFO      ━━━━  usa-brentwood (raster)  ━━━━
20:40:36  INFO      MEM [usa-brentwood raster start] RSS = 305 MB
20:40:37  INFO      [usa-brentwood] Raster CRS: EPSG:32610 | 16 tiles.
20:40:37  INFO      [usa-brentwood] Raster reference buildings: 2107 (from 1 file(s))
20:40:37  WARNING   [usa-brentwood / obt] No raster file found (pattern: usa_brentwood_obt_2023*).
20:40:37  INFO      [usa-brentwood / tempo_2023q4] Raster candidate: usa_brentwood_tempo_2023q4.tif
20:40:37  WARNING   [native-resolution guard] dataset='tempo' native_resolution_m=100.0, tolerance_factor=0.95, minimum_allowed_eval_resolution=95.000 m. Blocked finer evaluation grid(s): 10m (10.0 m) | Skipping blocked grid(s).
20:40:37  INFO      [usa-brentwood / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
20:40:37  INFO      MEM [usa-brentwood/tempo_2023q4 raster done] RSS = 334 MB
20:40:37  INFO      [usa-brentwood / ghsl_built_s_2025] Raster candidate: usa_brentwood_ghsl_bu

[zaf-durban_SN7.geojson] fixing 16 invalid geometries...


20:44:52  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:44:53  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:44:53  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:44:53  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:44:53  WARNING   CPLE_AppDefined:Value 0 in the source datase

[egy-cairo_SN7.geojson] fixing 1 invalid geometries...


20:45:06  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:45:06  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:45:07  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:45:07  WARNING   CPLE_AppDefined:Value 0 in the source dataset has been changed to 1.4013e-45 in the destination dataset to avoid being treated as NoData. To avoid this, select a different NoData value for the destination dataset.
20:45:07  WARNING   CPLE_AppDefined:Value 0 in the source datase

[chn-chengdu_SN7.geojson] fixing 1 invalid geometries...


20:48:05  INFO      [chn-chengdu / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
20:48:05  INFO      MEM [chn-chengdu/tempo_2023q4 raster done] RSS = 732 MB
20:48:05  INFO      [chn-chengdu / ghsl_built_s_2025] Raster candidate: chn_chengdu_ghsl_built_s_2025.tif
20:48:05  WARNING   [native-resolution guard] dataset='ghsl_built_s' native_resolution_m=100.0, tolerance_factor=0.95, minimum_allowed_eval_resolution=95.000 m. Blocked finer evaluation grid(s): 10m (10.0 m) | Skipping blocked grid(s).
20:48:07  INFO      [chn-chengdu / ghsl_built_s_2025] Raster tile metrics saved → raster_metrics_tiles_ghsl_built_s_2025.parquet
20:48:07  INFO      MEM [chn-chengdu/ghsl_built_s_2025 raster done] RSS = 732 MB
20:48:07  INFO      [chn-chengdu / wsf_tracker] Raster candidate: chn_chengdu_wsf_tracker.tif
20:48:13  INFO      [chn-chengdu / wsf_tracker] Raster tile metrics saved → raster_metrics_tiles_wsf_tracker.parquet
20:48:13  INFO      MEM [chn-chengdu/wsf_t


Done — 52 cities processed.



,count
status,
ok,52


,city,status
0,usa-brentwood,ok
1,usa-sandiego,ok
2,usa-boise,ok
3,usa-lasvegas,ok
4,usa-saltlakecity,ok
5,usa-permianbasin,ok
6,mex-mexicocity,ok
7,usa-bentonville,ok
8,usa-gonzales,ok
9,usa-atlanta,ok


In [12]:
summary

,city,status
0,usa-brentwood,ok
1,usa-sandiego,ok
2,usa-boise,ok
3,usa-lasvegas,ok
4,usa-saltlakecity,ok
5,usa-permianbasin,ok
6,mex-mexicocity,ok
7,usa-bentonville,ok
8,usa-gonzales,ok
9,usa-atlanta,ok
